# Análisis de movilidad urbana en Nueva York

Este proyecto analiza los viajes de Yellow Taxi de enero a septiembre
de 2026 para identificar patrones de demanda por horario y zona,
comparar los días de lunes a viernes con los fines de semana y
describir los cobros, las distancias y las duraciones de los viajes.

## 1. Carga e inspección de los datos

Se cargan nueve archivos Parquet mensuales. Se revisan el esquema,
el número de registros y el rango de fechas de recogida por archivo.
También se carga el catálogo de zonas de la TLC para identificar
los orígenes y destinos de los viajes.

**Fuentes**

- [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
- [Diccionario de datos de Yellow Taxi](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)

In [ ]:
import os
from pyspark.sql import DataFrame, SparkSession, functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

spark: SparkSession = (
    SparkSession.builder
    .appName("NYC Yellow Taxi 2026")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", os.environ.get("SPARK_DRIVER_HOST", "jupyter"))
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "3")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Master: {spark.sparkContext.master}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 02:37:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.1 | Master: spark://spark-master:7077


In [ ]:
from calendar import monthrange
from datetime import date, timedelta
from pathlib import Path

analysis_year: int = int(os.environ.get("TLC_ANALYSIS_YEAR", "2026"))
last_published_month: int = int(os.environ.get("TLC_LAST_MONTH", "9"))

source_months: tuple[str, ...] = tuple(
    f"{analysis_year}-{month:02d}"
    for month in range(1, last_published_month + 1)
)

first_pickup_day: date = date(analysis_year, 1, 1)
last_month_start: date = date(analysis_year, last_published_month, 1)
days_in_last_month: int = monthrange(analysis_year, last_published_month)[1]
end_exclusive: date = last_month_start + timedelta(days=days_in_last_month)

period_start: str = f"{first_pickup_day.isoformat()} 00:00:00"
period_end: str = f"{end_exclusive.isoformat()} 00:00:00"

raw_dir: Path = Path("/workspace/data/raw")
trip_paths: dict[str, Path] = {
    month: raw_dir / f"yellow_tripdata_{month}.parquet"
    for month in source_months
}

missing_files: list[str] = [
    path.name for path in trip_paths.values() if not path.is_file()
]
if missing_files:
    raise FileNotFoundError(f"Missing monthly files: {', '.join(missing_files)}")

trip_column_types: dict[str, str] = {
    "tpep_pickup_datetime": "timestamp_ntz",
    "tpep_dropoff_datetime": "timestamp_ntz",
    "PULocationID": "int",
    "DOLocationID": "int",
    "payment_type": "bigint",
    "trip_distance": "double",
    "fare_amount": "double",
    "total_amount": "double",
}


def read_monthly_trips(month: str, path: Path) -> DataFrame:
    monthly_trips: DataFrame = spark.read.parquet(str(path))
    missing_columns: set[str] = set(trip_column_types) - set(monthly_trips.columns)
    if missing_columns:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing_columns)}")

    return monthly_trips.select(
        F.lit(month).alias("source_month"),
        *[
            F.col(column).cast(sql_type).alias(column)
            for column, sql_type in trip_column_types.items()
        ],
    )


monthly_frames: list[DataFrame] = [
    read_monthly_trips(month, path) for month, path in trip_paths.items()
]

trips_raw: DataFrame = monthly_frames[0]
for monthly_trips in monthly_frames[1:]:
    trips_raw = trips_raw.unionByName(monthly_trips)

zone_schema: StructType = StructType([
    StructField("LocationID", IntegerType()),
    StructField("Borough", StringType()),
    StructField("Zone", StringType()),
    StructField("service_zone", StringType()),
])

zones: DataFrame = (
    spark.read
    .option("header", True)
    .option("enforceSchema", False)
    .option("mode", "FAILFAST")
    .schema(zone_schema)
    .csv(str(raw_dir / "taxi_zone_lookup.csv"))
)

trip_profile: DataFrame = (
    trips_raw.groupBy("source_month")
    .agg(
        F.count("*").alias("raw_trip_count"),
        F.min("tpep_pickup_datetime").alias("earliest_pickup"),
        F.max("tpep_pickup_datetime").alias("latest_pickup"),
    )
    .orderBy("source_month")
)

print(f"Pickup period: {period_start} through {period_end} (exclusive)")
trips_raw.printSchema()
trip_profile.show(len(source_months), truncate=False)
print(f"Zone catalog rows: {zones.count():,}")

Pickup period: 2026-01-01 00:00:00 through 2026-10-01 00:00:00 (exclusive)
root
 |-- source_month: string (nullable = false)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)



+------------+--------------+-------------------+-------------------+
|source_month|raw_trip_count|earliest_pickup    |latest_pickup      |
+------------+--------------+-------------------+-------------------+
|2026-01     |3724889       |2025-12-31 23:57:29|2026-02-01 00:45:01|
|2026-02     |3399866       |2026-01-31 23:31:23|2026-03-01 00:51:48|
|2026-03     |3952451       |2008-12-31 23:03:20|2026-04-01 00:06:25|
|2026-04     |3831240       |2001-01-01 09:23:58|2026-05-01 00:01:28|
|2026-05     |4090836       |2008-12-31 23:05:53|2026-06-01 00:20:35|
|2026-06     |3837248       |2008-12-31 23:03:25|2026-06-30 23:59:59|
|2026-07     |3530109       |2008-12-30 23:06:00|2026-08-05 20:54:00|
|2026-08     |3336716       |2009-01-01 14:39:49|2026-08-31 23:59:59|
|2026-09     |3876044       |2009-01-01 09:32:44|2026-10-07 07:40:40|
+------------+--------------+-------------------+-------------------+

Zone catalog rows: 265


## 2. Limpieza y preparación de los viajes

El periodo se define por la fecha de recogida. Se incluyen los viajes
que comenzaron desde el 1 de enero de 2026 a las 00:00 hasta antes
del 1 de octubre de 2026 a las 00:00.

Cada registro recibe un estado de calidad. Se excluyen las recogidas
fuera del periodo, las fechas faltantes, las duraciones no positivas,
las distancias faltantes o inválidas y las tarifas que no se pueden
representar como un importe decimal.

Entre los viajes que pasan esos filtros, se calcula el percentil 99.9
de duración y distancia. Los viajes que superan cualquiera de esos
límites se excluyen de la base analítica. Si un viaje cumple varias
condiciones de exclusión, se contabiliza bajo la primera.

Para los viajes aceptados se calcula la hora de recogida, el día de la
semana, la duración en minutos y la distancia en kilómetros.

In [3]:
from pyspark.sql import Column, Row

start_at: Column = F.lit(period_start).cast("timestamp_ntz")
end_at: Column = F.lit(period_end).cast("timestamp_ntz")
pickup_at: Column = F.col("tpep_pickup_datetime")
dropoff_at: Column = F.col("tpep_dropoff_datetime")
distance_miles: Column = F.col("trip_distance")
fare_for_quality: Column = F.col("fare_amount").try_cast("decimal(18,2)")

duration_for_quality: Column = (
    F.timestamp_diff("MICROSECOND", pickup_at, dropoff_at) / 60_000_000
)

initial_status: Column = (
    F.when(pickup_at.isNull(), "missing_pickup")
    .when((pickup_at < start_at) | (pickup_at >= end_at), "outside_period")
    .when(dropoff_at.isNull(), "missing_dropoff")
    .when(dropoff_at <= pickup_at, "nonpositive_duration")
    .when(distance_miles.isNull(), "missing_distance")
    .when(
        F.isnan(distance_miles)
        | (distance_miles <= 0)
        | (distance_miles >= F.lit(float("inf"))),
        "invalid_distance",
    )
    .when(fare_for_quality.isNull(), "missing_or_invalid_fare")
    .otherwise("valid")
)

trips_screened: DataFrame = trips_raw.withColumn("quality_status", initial_status)
eligible_trips: DataFrame = trips_screened.filter(F.col("quality_status") == "valid")

outlier_percentile: float = 0.999
outlier_limits: Row = eligible_trips.agg(
    F.percentile_approx(duration_for_quality, outlier_percentile, 20_000).alias(
        "duration_limit_min"
    ),
    F.percentile_approx(distance_miles, outlier_percentile, 20_000).alias(
        "distance_limit_miles"
    ),
).collect()[0]

duration_limit_min: float | None = outlier_limits["duration_limit_min"]
distance_limit_miles: float | None = outlier_limits["distance_limit_miles"]

if duration_limit_min is None or distance_limit_miles is None:
    raise ValueError("No eligible trips available to calculate outlier limits")

current_status: Column = F.col("quality_status")
trips_classified: DataFrame = trips_screened.withColumn(
    "quality_status",
    F.when(current_status != "valid", current_status)
    .when(duration_for_quality > duration_limit_min, "duration_outlier")
    .when(distance_miles > distance_limit_miles, "distance_outlier")
    .otherwise("valid"),
)

quality_counts: DataFrame = (
    trips_classified.groupBy("quality_status")
    .agg(F.count("*").alias("trip_count"))
    .orderBy(F.desc("trip_count"))
)

print(f"P99.9 duration limit: {duration_limit_min:.2f} minutes")
print(f"P99.9 distance limit: {distance_limit_miles:.2f} miles")
quality_counts.show(truncate=False)

P99.9 duration limit: 117.38 minutes
P99.9 distance limit: 31.07 miles


+--------------------+----------+
|quality_status      |trip_count|
+--------------------+----------+
|valid               |32009865  |
|invalid_distance    |1087317   |
|nonpositive_duration|419885    |
|duration_outlier    |32254     |
|distance_outlier    |30041     |
|outside_period      |37        |
+--------------------+----------+



In [4]:
miles_to_kilometers: float = 1.609344


trips_clean: DataFrame = (
    trips_classified.filter(F.col("quality_status") == "valid")
    .drop("quality_status")
    .withColumn("pickup_date", F.to_date(pickup_at))
    .withColumn("pickup_hour", F.hour(pickup_at))
    .withColumn("iso_weekday", F.weekday(pickup_at) + 1)
    .withColumn(
        "day_type",
        F.when(F.col("iso_weekday") >= 6, "weekend").otherwise("weekday"),
    )
    .withColumn("duration_minutes", duration_for_quality)
    .withColumn("distance_km", distance_miles * miles_to_kilometers)
)

clean_profile: DataFrame = trips_clean.agg(
    F.count("*").alias("clean_trip_count"),
    F.min("pickup_date").alias("first_pickup_date"),
    F.max("pickup_date").alias("last_pickup_date"),
    F.count_if(F.col("distance_km").isNull()).alias("trip_without_distance"),
    F.round(F.percentile_approx("duration_minutes", 0.99), 2).alias(
        "p99_duration_min"
    ),
    F.round(F.max("duration_minutes"), 2).alias("max_duration_min")
)

clean_profile.show(truncate=False)

+----------------+-----------------+----------------+---------------------+----------------+----------------+
|clean_trip_count|first_pickup_date|last_pickup_date|trip_without_distance|p99_duration_min|max_duration_min|
+----------------+-----------------+----------------+---------------------+----------------+----------------+
|32009865        |2026-01-01       |2026-09-30      |0                    |69.42           |117.38          |
+----------------+-----------------+----------------+---------------------+----------------+----------------+



## 3. Enriquecimiento con zonas de taxi
Se une el catálogo de la TLC con los viajes mediante dos uniones izquierdas. 
La primera usa `PULocationID` para el origen y la segunda usa `DOLocationID` 
para el destino. Cada unión agrega el borough, el nombre de la zona y la 
zona de servicio.

La validación comprueba que el catálogfo tenga claves únicas y que el total 
de viajes se conserve. Para cada extremo del viaje se registra si la zona 
está identificada, si el catálogo la marca como desconocida o fuera de 
Nueva York, si falta el identificador o si este carece de coincidencia.

In [5]:
invalid_zone_keys: DataFrame =(
    zones.groupBy("LocationID")
    .count()
    .filter(F.col("LocationID").isNull() | (F.col("count") > 1))
)
assert not invalid_zone_keys.take(1), "Invalid or duplicate zone catalog keys"

pickup_zones: DataFrame = zones.select(
    F.col("LocationID").alias("pickup_catalog_id"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone"),
    F.col("service_zone").alias("pickup_service_zone"),
)

dropoff_zones: DataFrame = zones.select(
    F.col("LocationID").alias("dropoff_catalog_id"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone"),
    F.col("service_zone").alias("dropoff_service_zone")
)

def classify_zone(location_column: str, catalog_column: str) -> Column:
    location_id: Column = F.col(location_column)
    return (
        F.when(location_id.isNull(), "missing_id")
        .when(F.col(catalog_column).isNull(), "not_in_catalog")
        .when(location_id == 264, "unknown_zone")
        .when(location_id == 265, "outside_nyc")
        .otherwise("mapped_zone")
    )

trips_enriched: DataFrame = (
    trips_clean.join(
        F.broadcast(pickup_zones),
        F.col("PULocationID") == F.col("pickup_catalog_id"),
        "left"
    )
    .join(
        F.broadcast(dropoff_zones),
        F.col("DOLocationID") == F.col("dropoff_catalog_id"),
        "left"
    )
    .withColumn("pickup_zone_status", classify_zone("PULocationID", "pickup_catalog_id"))
    .withColumn("dropoff_zone_status", classify_zone("DOLocationID", "dropoff_catalog_id"))
    .drop("pickup_catalog_id", "dropoff_catalog_id")
)

In [6]:
from pyspark.sql import Row

zone_statuses: tuple[str, ...] = (
    "mapped_zone",
    "unknown_zone",
    "outside_nyc",
    "missing_id",
    "not_in_catalog",
)

zone_audit: Row = trips_enriched.agg(
    F.count("*").alias("enriched_trip_count"),
    *[
        F.count_if(F.col(f"{side}_zone_status") == status).alias(f"{side}_{status}")
        for side in ("pickup", "dropoff")
        for status in zone_statuses
    ],
).collect()[0]

clean_trip_count: int = trips_clean.count()
enriched_trip_count: int = zone_audit["enriched_trip_count"]

assert enriched_trip_count == clean_trip_count, "Zone joins changed the trip count"

print(f"Trips before joins: {clean_trip_count:,}")
print(f"Trips after joins:  {enriched_trip_count:,}")
print()
print(f"{'Zone status':<16} {'Pickup':>12} {'Dropoff':>12}")
for status in zone_statuses:
    print(f"{status:<16} {zone_audit[f'pickup_{status}']:>12,} "
          f"{zone_audit[f'dropoff_{status}']:>12,}")

trips_enriched.select(
    "PULocationID", "pickup_borough", "pickup_zone",
    "DOLocationID", "dropoff_borough", "dropoff_zone",
).show(5, truncate=False)

Trips before joins: 32,009,865
Trips after joins:  32,009,865

Zone status            Pickup      Dropoff
mapped_zone        31,970,468   31,851,778
unknown_zone           32,458       34,392
outside_nyc             6,939      123,695
missing_id                  0            0
not_in_catalog              0            0
+------------+--------------+------------------------+------------+---------------+---------------------+
|PULocationID|pickup_borough|pickup_zone             |DOLocationID|dropoff_borough|dropoff_zone         |
+------------+--------------+------------------------+------------+---------------+---------------------+
|239         |Manhattan     |Upper West Side South   |238         |Manhattan      |Upper West Side North|
|163         |Manhattan     |Midtown North           |162         |Manhattan      |Midtown East         |
|43          |Manhattan     |Central Park            |237         |Manhattan      |Upper East Side South|
|142         |Manhattan     |Lincoln Square

## 4. Demanda por horario y zona

La demanda se mide contando los viajes válidos según su hora de recogida.
Para comparar los días de lunes a viernes con los fines de semana, se
divide el número de viajes entre los días de cada grupo en el calendario.
Para el perfil horario se incluyen las 24 horas, incluso cuando alguna
combinación de hora y tipo de día tiene cero viajes.

Los rankings de origen y destino se calculan por separado. Cada ranking
incluye las zonas identificadas para ese extremo del viaje. Se conservan
todos los viajes válidos para los indicadores de demanda por horario.

In [7]:
calendar_days: DataFrame = spark.range(1).select(
    F.explode(
        F.sequence(F.to_date(start_at), F.date_sub(F.to_date(end_at), 1))
    ).alias("pickup_date")
)

calendar_day_counts: DataFrame = (
    calendar_days.withColumn(
        "day_type",
        F.when(F.dayofweek("pickup_date").isin(1, 7), "weekend")
        .otherwise("weekday"),
    )
    .groupBy("day_type")
    .agg(F.count("*").alias("day_count"))
)

hours: DataFrame = spark.range(24).select(
    F.col("id").cast("int").alias("pickup_hour")
)

observed_hourly_demand: DataFrame = trips_clean.groupBy(
    "day_type", "pickup_hour"
).agg(F.count("*").alias("trip_count"))

demand_by_day_type_hour: DataFrame = (
    calendar_day_counts.crossJoin(hours)
    .join(observed_hourly_demand, ["day_type", "pickup_hour"], "left")
    .fillna({"trip_count": 0})
    .withColumn(
        "avg_trips_per_day",
        F.round(F.col("trip_count") / F.col("day_count"), 2),
    )
)

demand_by_day_type: DataFrame = (
    demand_by_day_type_hour.groupBy("day_type", "day_count")
    .agg(F.sum("trip_count").alias("trip_count"))
    .withColumn(
        "avg_trips_per_day",
        F.round(F.col("trip_count") / F.col("day_count"), 2),
    )
)

demand_by_hour: DataFrame = demand_by_day_type_hour.groupBy("pickup_hour").agg(
    F.sum("trip_count").alias("trip_count")
)

demand_by_day_type.orderBy("day_type").show(truncate=False)
demand_by_hour.orderBy(F.desc("trip_count"), "pickup_hour").show(5, truncate=False)

+--------+---------+----------+-----------------+
|day_type|day_count|trip_count|avg_trips_per_day|
+--------+---------+----------+-----------------+
|weekday |195      |22713121  |116477.54        |
|weekend |78       |9296744   |119189.03        |
+--------+---------+----------+-----------------+



+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|18         |2070915   |
|17         |1996454   |
|21         |1954604   |
|19         |1898253   |
|20         |1896160   |
+-----------+----------+
only showing top 5 rows


In [8]:
def count_trips_by_zone(
    trips: DataFrame, side: str, location_column: str
) -> DataFrame:
    return (
        trips.filter(F.col(f"{side}_zone_status") == "mapped_zone")
        .select(
            F.col(location_column).alias("location_id"),
            F.col(f"{side}_borough").alias("borough"),
            F.col(f"{side}_zone").alias("zone"),
        )
        .groupBy("location_id", "borough", "zone")
        .agg(F.count("*").alias("trip_count"))
    )


pickup_zone_counts: DataFrame = count_trips_by_zone(
    trips_enriched, "pickup", "PULocationID"
)
dropoff_zone_counts: DataFrame = count_trips_by_zone(
    trips_enriched, "dropoff", "DOLocationID"
)

top_pickup_zones: DataFrame = pickup_zone_counts.orderBy(
    F.desc("trip_count"), "location_id"
).limit(10)

top_dropoff_zones: DataFrame = dropoff_zone_counts.orderBy(
    F.desc("trip_count"), "location_id"
).limit(10)

print("Top 10 pickup zones")
top_pickup_zones.show(10, truncate=False)

print("Top 10 dropoff zones")
top_dropoff_zones.show(10, truncate=False)

Top 10 pickup zones


+-----------+---------+----------------------------+----------+
|location_id|borough  |zone                        |trip_count|
+-----------+---------+----------------------------+----------+
|237        |Manhattan|Upper East Side South       |1424065   |
|161        |Manhattan|Midtown Center              |1316698   |
|132        |Queens   |JFK Airport                 |1277487   |
|236        |Manhattan|Upper East Side North       |1264648   |
|186        |Manhattan|Penn Station/Madison Sq West|988484    |
|162        |Manhattan|Midtown East                |975089    |
|230        |Manhattan|Times Sq/Theatre District   |923127    |
|142        |Manhattan|Lincoln Square East         |921531    |
|79         |Manhattan|East Village                |858835    |
|170        |Manhattan|Murray Hill                 |834825    |
+-----------+---------+----------------------------+----------+

Top 10 dropoff zones


+-----------+---------+-------------------------+----------+
|location_id|borough  |zone                     |trip_count|
+-----------+---------+-------------------------+----------+
|236        |Manhattan|Upper East Side North    |1305879   |
|237        |Manhattan|Upper East Side South    |1279702   |
|161        |Manhattan|Midtown Center           |1088085   |
|170        |Manhattan|Murray Hill              |883977    |
|230        |Manhattan|Times Sq/Theatre District|873205    |
|162        |Manhattan|Midtown East             |833364    |
|239        |Manhattan|Upper West Side South    |824532    |
|142        |Manhattan|Lincoln Square East      |816642    |
|68         |Manhattan|East Chelsea             |802196    |
|141        |Manhattan|Lenox Hill West          |796807    |
+-----------+---------+-------------------------+----------+



## 5. Cobros y características de los viajes

Se analizan la tarifa registrada y el cobro total en dólares por zona
de origen, día de la semana y tipo de pago. Las sumas conservan los
importes negativos y se reporta cuántos registros los contienen.
Los importes ausentes se reflejan en el conteo de valores disponibles.

La duración y la distancia se resumen por hora y zona de recogida.
Se reportan el promedio y la mediana aproximada porque el perfil inicial
mostró duraciones extremas. El promedio de distancia usa las distancias
válidas y se indica su tamaño de muestra. También se comparan los viajes
que salen de Newark, JFK y LaGuardia.

El cobro promedio por viaje se calcula con los registros que tienen un 
cobro total disponible.

In [9]:
from operator import itemgetter
payment_labels: dict = {
    0: "Flex Fare",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip"
}

payment_lookup: Column = F.create_map(
    *[
        item
        for code, label in payment_labels.items()
        for item in (F.lit(code).cast("long"), F.lit(label))
    ]
)

payment_code: Column = F.col("payment_type")

trips_with_amounts: DataFrame = (
    trips_enriched.withColumn(
        "payment_method",
        F.when(payment_code.isNull(), "Missing code").otherwise(
            F.coalesce(F.element_at(payment_lookup, payment_code), F.lit("Unmapped code"))
        ),
    )
    .withColumn("fare_usd", F.col("fare_amount").try_cast("decimal(18,2)"))
    .withColumn("total_usd", F.col("total_amount").try_cast("decimal(18,2)"))
)

def summarize_amounts(trips: DataFrame, *group_columns: str) -> DataFrame:
    return trips.groupBy(*group_columns).agg(
        F.count("*").alias("trip_count"),
        F.count("total_usd").alias("trips_with_total"),
        F.sum("fare_usd").alias("fare_sum_usd"),
        F.sum("total_usd").alias("reported_total_usd"),
        F.round(F.avg("total_usd"), 2).alias("avg_reported_total_usd"),
        F.count_if(F.col("total_usd") < 0).alias("negative_total_count"),
    )

amounts_by_pickup_zone: DataFrame = summarize_amounts(
    trips_with_amounts.filter(F.col("pickup_zone_status") == "mapped_zone"),
    "PULocationID", "pickup_borough", "pickup_zone",
)

amounts_by_weekday: DataFrame = summarize_amounts(
    trips_with_amounts, "iso_weekday"
)

amounts_by_payment: DataFrame = summarize_amounts(
    trips_with_amounts, "payment_type", "payment_method"
)

print("Top 10 pickup zones by reported total")
amounts_by_pickup_zone.orderBy(
    F.desc("reported_total_usd"), "PULocationID"
).show(10, truncate=False)

print("Reported amounts by ISO weekday (1 = Monday, 7 = Sunday)")
amounts_by_weekday.orderBy("iso_weekday").show(7, truncate=False)

print("Reported amounts by payment method")
amounts_by_payment.orderBy(F.desc("trip_count"), "payment_type").show(
    truncate=False
)

Top 10 pickup zones by reported total


+------------+--------------+----------------------------+----------+----------------+------------+------------------+----------------------+--------------------+
|PULocationID|pickup_borough|pickup_zone                 |trip_count|trips_with_total|fare_sum_usd|reported_total_usd|avg_reported_total_usd|negative_total_count|
+------------+--------------+----------------------------+----------+----------------+------------+------------------+----------------------+--------------------+
|132         |Queens        |JFK Airport                 |1277487   |1277487         |73960829.83 |96137836.10       |75.26                 |23145               |
|138         |Queens        |LaGuardia Airport           |818000    |818000          |35882406.40 |56530430.22       |69.11                 |6904                |
|161         |Manhattan     |Midtown Center              |1316698   |1316698         |23650178.12 |35850232.86       |27.23                 |6857                |
|237         |Manhatta

+-----------+----------+----------------+------------+------------------+----------------------+--------------------+
|iso_weekday|trip_count|trips_with_total|fare_sum_usd|reported_total_usd|avg_reported_total_usd|negative_total_count|
+-----------+----------+----------------+------------+------------------+----------------------+--------------------+
|1          |3781437   |3781437         |80446038.26 |115870495.97      |30.64                 |19435               |
|2          |4437083   |4437083         |92849797.23 |133826643.12      |30.16                 |21824               |
|3          |4733045   |4733045         |99536759.52 |143257634.96      |30.27                 |22734               |
|4          |5004854   |5004854         |107073335.86|153117621.63      |30.59                 |24430               |
|5          |4756702   |4756702         |99496899.28 |142063865.06      |29.87                 |24755               |
|6          |5060880   |5060880         |104541882.99|14

+------------+--------------+----------+----------------+------------+------------------+----------------------+--------------------+
|payment_type|payment_method|trip_count|trips_with_total|fare_sum_usd|reported_total_usd|avg_reported_total_usd|negative_total_count|
+------------+--------------+----------+----------------+------------+------------------+----------------------+--------------------+
|1           |Credit card   |20687398  |20687398        |405040823.84|618787487.32      |29.91                 |2310                |
|0           |Flex Fare     |8117765   |8117765         |212896084.10|266784381.19      |32.86                 |116                 |
|2           |Cash          |2894907   |2894907         |56222999.91 |72450899.30       |25.03                 |38764               |
|4           |Dispute       |231348    |231348          |504951.39   |663445.89         |2.87                  |101714              |
|3           |No charge     |78444     |78444           |78102

In [10]:
def summarize_trip_metrics(trips: DataFrame, *group_columns: str) -> DataFrame:
    return trips.groupBy(*group_columns).agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("duration_minutes"), 2).alias("avg_duration_min"),
        F.round(F.percentile_approx("duration_minutes", 0.5), 2).alias(
            "median_duration_min"
        ),
        F.count("distance_km").alias("trips_with_distance"),
        F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
        F.round(F.percentile_approx("distance_km", 0.5), 2).alias(
            "median_distance_km"
        ),
    )


trip_metrics_by_hour: DataFrame = summarize_trip_metrics(
    trips_clean, "pickup_hour"
)

trip_metrics_by_pickup_zone: DataFrame = summarize_trip_metrics(
    trips_enriched.filter(F.col("pickup_zone_status") == "mapped_zone"),
    "PULocationID", "pickup_borough", "pickup_zone",
)

airport_pickup_ids: tuple[int, ...] = (1, 132, 138)

airport_pickup_metrics: DataFrame = trip_metrics_by_pickup_zone.filter(
    F.col("PULocationID").isin(*airport_pickup_ids)
)

print("Duration and distance by pickup hour")
trip_metrics_by_hour.orderBy("pickup_hour").show(24, truncate=False)

print("Duration and distance for the 10 busiest pickup zones")
trip_metrics_by_pickup_zone.orderBy(
    F.desc("trip_count"), "PULocationID"
).show(10, truncate=False)

print("Duration and distance for pickups at Newark, JFK and LaGuardia")
airport_pickup_metrics.orderBy(F.desc("trip_count")).show(truncate=False)

Duration and distance by pickup hour


+-----------+----------+----------------+-------------------+-------------------+---------------+------------------+
|pickup_hour|trip_count|avg_duration_min|median_duration_min|trips_with_distance|avg_distance_km|median_distance_km|
+-----------+----------+----------------+-------------------+-------------------+---------------+------------------+
|0          |1039138   |15.75           |13.93              |1039138            |6.61           |4.25              |
|1          |686145    |14.88           |13.25              |686145             |6.28           |4.18              |
|2          |457712    |14.26           |12.87              |457712             |6.17           |4.31              |
|3          |327807    |14.36           |12.87              |327807             |6.68           |4.67              |
|4          |268504    |15.74           |13.77              |268504             |8.26           |5.75              |
|5          |297693    |17.57           |13.65              |297

+------------+--------------+----------------------------+----------+----------------+-------------------+-------------------+---------------+------------------+
|PULocationID|pickup_borough|pickup_zone                 |trip_count|avg_duration_min|median_duration_min|trips_with_distance|avg_distance_km|median_distance_km|
+------------+--------------+----------------------------+----------+----------------+-------------------+-------------------+---------------+------------------+
|237         |Manhattan     |Upper East Side South       |1424065   |12.67           |10.03              |1424065            |2.94           |2.0               |
|161         |Manhattan     |Midtown Center              |1316698   |16.0            |13.72              |1316698            |3.77           |2.51              |
|132         |Queens        |JFK Airport                 |1277487   |40.59           |40.02              |1277487            |23.56          |27.09             |
|236         |Manhattan     

+------------+--------------+-----------------+----------+----------------+-------------------+-------------------+---------------+------------------+
|PULocationID|pickup_borough|pickup_zone      |trip_count|avg_duration_min|median_duration_min|trips_with_distance|avg_distance_km|median_distance_km|
+------------+--------------+-----------------+----------+----------------+-------------------+-------------------+---------------+------------------+
|132         |Queens        |JFK Airport      |1277487   |40.59           |40.02              |1277487            |23.56          |27.09             |
|138         |Queens        |LaGuardia Airport|818000    |32.04           |29.55              |818000             |15.33          |15.06             |
|1           |EWR           |Newark Airport   |1011      |3.9             |0.15               |1011               |2.54           |0.08              |
+------------+--------------+-----------------+----------+----------------+-------------------

In [11]:
trips_by_pickup_month: DataFrame = (
    trips_clean.withColumn("pickup_month", F.date_format("pickup_date", "yyyy-MM"))
    .groupBy("pickup_month")
    .agg(F.count("*").alias("trip_count"))
    .orderBy("pickup_month")
)

monthly_rows: list[Row] = trips_by_pickup_month.collect()
observed_months: tuple[str, ...] = tuple(row["pickup_month"] for row in monthly_rows)
monthly_total: int = sum(row["trip_count"] for row in monthly_rows)

assert observed_months == source_months, f"Unexpected pickup months: {observed_months}"
assert monthly_total == clean_trip_count, "Monthly counts do not match clean trips"

for row in monthly_rows:
    print(f"{row['pickup_month']}  {row['trip_count']:>12,}")

print(f"Total valid trips: {monthly_total:,}")

2026-01     3,547,578
2026-02     3,230,079
2026-03     3,775,606
2026-04     3,680,681
2026-05     3,917,300
2026-06     3,655,473
2026-07     3,354,894
2026-08     3,172,595
2026-09     3,675,659
Total valid trips: 32,009,865


## 6. Exportación de indicadores

Se prepara un documento por periodo con los conteos de calidad, la
demanda, los rankings de zonas, los cobros y las métricas de viaje.
El conteo por zona de recogida y hora alimenta el mapa de calor del
dashboard. Antes de exportar, se verifica que los indicadores de
cobertura general coincidan con el total de viajes válidos.

Los importes se exportan en JSON extendido para conservar su precisión
decimal al cargarlos en MongoDB.

In [ ]:
import json
from decimal import Decimal
from tempfile import NamedTemporaryFile

demand_by_zone_hour: DataFrame = (
    trips_enriched.filter(F.col("pickup_zone_status") == "mapped_zone")
    .groupBy("PULocationID", "pickup_borough", "pickup_zone", "pickup_hour")
    .agg(F.count("*").alias("trip_count"))
    .orderBy("PULocationID", "pickup_hour")
)

indicator_frames: dict[str, DataFrame] = {
    "quality_counts": quality_counts,
    "demand_by_hour": demand_by_hour.orderBy("pickup_hour"),
    "demand_by_day_type": demand_by_day_type.orderBy("day_type"),
    "demand_by_day_type_hour": demand_by_day_type_hour.orderBy(
        "day_type", "pickup_hour"
    ),
    "demand_by_zone_hour": demand_by_zone_hour,
    "top_pickup_zones": top_pickup_zones,
    "top_dropoff_zones": top_dropoff_zones,
    "amounts_by_pickup_zone": amounts_by_pickup_zone.orderBy(
        F.desc("reported_total_usd"), "PULocationID"
    ),
    "amounts_by_weekday": amounts_by_weekday.orderBy("iso_weekday"),
    "amounts_by_payment": amounts_by_payment.orderBy(F.desc("trip_count")),
    "trip_metrics_by_hour": trip_metrics_by_hour.orderBy("pickup_hour"),
    "trip_metrics_by_pickup_zone": trip_metrics_by_pickup_zone.orderBy(
        F.desc("trip_count"), "PULocationID"
    ),
    "airport_pickup_metrics": airport_pickup_metrics.orderBy(F.desc("trip_count")),
}

indicator_rows: dict[str, list[Row]] = {
    name: frame.collect() for name, frame in indicator_frames.items()
}
indicator_rows["trips_by_pickup_month"] = monthly_rows

for name in (
    "demand_by_hour",
    "demand_by_day_type",
    "demand_by_day_type_hour",
    "amounts_by_weekday",
    "amounts_by_payment",
    "trip_metrics_by_hour",
):
    indicator_total: int = sum(row["trip_count"] for row in indicator_rows[name])
    if indicator_total != monthly_total:
        raise ValueError(f"{name}: expected {monthly_total}, got {indicator_total}")

heatmap_total: int = sum(
    row["trip_count"] for row in indicator_rows["demand_by_zone_hour"]
)
if heatmap_total != zone_audit["pickup_mapped_zone"]:
    raise ValueError("Heatmap counts do not match trips with mapped pickup zones")

report_id: str = f"{source_months[0]}_{source_months[-1]}"
report: dict[str, object] = {
    "_id": report_id,
    "period_start": first_pickup_day.isoformat(),
    "period_end_exclusive": end_exclusive.isoformat(),
    "source_months": list(source_months),
    "valid_trip_count": monthly_total,
    "raw_trip_count": sum(row["trip_count"] for row in indicator_rows["quality_counts"]),
    "duration_limit_min": duration_limit_min,
    "distance_limit_miles": distance_limit_miles,
    "indicators": {
        name: [row.asDict(recursive=True) for row in rows]
        for name, rows in indicator_rows.items()
    },
}


def encode_decimal(value: object) -> dict[str, str]:
    if isinstance(value, Decimal):
        return {"$numberDecimal": str(value)}
    raise TypeError(f"Unsupported value: {type(value).__name__}")


report_path: Path = Path(
    os.environ.get(
        "MOBILITY_REPORT_PATH",
        "/workspace/data/processed/mobility_report.json",
    )
)
report_path.parent.mkdir(parents=True, exist_ok=True)

temporary_file = NamedTemporaryFile(
    mode="w",
    encoding="utf-8",
    dir=report_path.parent,
    prefix=".mobility_report-",
    suffix=".tmp",
    delete=False,
)
temporary_path: Path = Path(temporary_file.name)

try:
    with temporary_file:
        json.dump(report, temporary_file, default=encode_decimal, allow_nan=False, indent=2, ensure_ascii=False)
        temporary_file.write("\n")
    temporary_path.chmod(0o644)
    temporary_path.replace(report_path)
finally:
    temporary_path.unlink(missing_ok=True)

print(f"Report ID: {report_id}")
print(f"Valid trips: {monthly_total:,}")
print(f"Indicator tables: {len(indicator_rows)}")
print(f"Saved to: {report_path}")
spark.stop()

Report ID: 2026-01_2026-09
Valid trips: 32,009,865
Indicator tables: 14
Saved to: /workspace/data/processed/mobility_report.json
